# CasCrop: Full Experiment Pipeline
## Crop Waste as Economic Contagion — Graph Neural Network Experiments

This notebook runs **all experiments** needed for publication:
1. Environment setup + data download (~15 min)
2. Data processing + graph construction (~5 min)
3. Main ablation: 5 models × 5 seeds × 200 epochs (~4-6 hrs on Colab GPU)
4. Extra experiments: graph perturbation, edge ablation, disentanglement probe (~1 hr)
5. Per-crop and per-cause subgroup analysis (~5 min)
6. Statistical significance tests (~5 min)
7. Case study cascade reconstruction (~10 min)
8. Publication figures + LaTeX tables (~5 min)

**Total runtime: ~6-8 hours on Colab T4 GPU**

---

## 0. Setup Environment

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# Clone repo and install dependencies
!git clone https://github.com/keshavkrishnan08/CasCrop.git
%cd CasCrop
!pip install -q pandas pyarrow scipy scikit-learn statsmodels seaborn geopandas shapely tqdm pyyaml

In [ ]:
import sys, os
sys.path.insert(0, 'src')
os.makedirs('data/raw/rma', exist_ok=True)
os.makedirs('data/raw/nass', exist_ok=True)
os.makedirs('data/raw/weather', exist_ok=True)
os.makedirs('data/raw/prices', exist_ok=True)
os.makedirs('data/raw/geographic', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/graphs', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('paper/figures', exist_ok=True)
os.makedirs('paper/tables', exist_ok=True)
print('Directory structure ready')

## 1. Download All Data (~15 min)
All datasets are freely available — no API keys needed.

In [ ]:
import requests, zipfile, gzip, io, time
from pathlib import Path

def download(url, path, desc=""):
    """Download a file with retry logic."""
    path = Path(path)
    if path.exists() and path.stat().st_size > 100:
        print(f"  Already exists: {path.name}")
        return True
    headers = {'User-Agent': 'CasCrop-Research/1.0'}
    for attempt in range(3):
        try:
            r = requests.get(url, headers=headers, timeout=120, stream=True)
            r.raise_for_status()
            # Check for HTML error pages
            first_chunk = next(r.iter_content(1024))
            if first_chunk.strip().startswith(b'<!DOCTYPE') or first_chunk.strip().startswith(b'<html'):
                print(f"  HTML error page, skipping: {desc}")
                return False
            with open(path, 'wb') as f:
                f.write(first_chunk)
                for chunk in r.iter_content(8192):
                    f.write(chunk)
            print(f"  Downloaded: {path.name} ({path.stat().st_size:,} bytes)")
            return True
        except Exception as e:
            if attempt < 2: time.sleep(2**attempt)
    print(f"  FAILED: {desc}")
    return False

print("Download function ready")

In [ ]:
%%time
# 1a. USDA RMA Cause of Loss Data (training labels)
print("=== USDA RMA Crop Insurance Claims ===")
rma_base = "https://pubfs-rma.fpac.usda.gov/pub/Web_Data_Files/Summary_of_Business/cause_of_loss/colsom_{year}.zip"
for year in range(2015, 2026):
    url = rma_base.format(year=year)
    zip_path = Path(f'data/raw/rma/colsom_{year}.zip')
    txt_path = Path(f'data/raw/rma/colsom_{year}.txt')
    if txt_path.exists() and txt_path.stat().st_size > 1000:
        print(f"  Already exists: colsom_{year}.txt")
        continue
    if download(url, zip_path, f"RMA {year}"):
        try:
            with zipfile.ZipFile(zip_path, 'r') as zf:
                zf.extractall('data/raw/rma/')
            print(f"  Extracted: colsom_{year}")
        except zipfile.BadZipFile:
            print(f"  Bad zip: {year}")

In [ ]:
%%time
# 1b. Geographic data (county adjacency + gazetteer + shapefile)
print("=== Geographic Data ===")
download('https://www2.census.gov/geo/docs/reference/county_adjacency.txt',
         'data/raw/geographic/county_adjacency.txt', 'County adjacency')

for yr in ['2023', '2022', '2021']:
    url = f'https://www2.census.gov/geo/docs/maps-data/data/gazetteer/{yr}_Gazetteer/{yr}_Gaz_counties_national.zip'
    zpath = Path(f'data/raw/geographic/county_gazetteer_{yr}.zip')
    if download(url, zpath, f'Gazetteer {yr}'):
        with zipfile.ZipFile(zpath, 'r') as zf:
            zf.extractall('data/raw/geographic/')
        break

In [ ]:
%%time
# 1c. Commodity prices from FRED
print("=== Commodity Prices ===")
price_urls = {
    'corn': 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=PMAIZMTUSDM',
    'wheat': 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=PWHEAMTUSDM',
    'soybeans': 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=PSOYBUSDM',
}
for crop, url in price_urls.items():
    download(url, f'data/raw/prices/{crop}_prices.csv', f'{crop} prices')

In [ ]:
%%time
# 1d. NOAA county-level climate data (nClimDiv)
print("=== NOAA Climate Data ===")
# Find the latest date stamp
r = requests.get('https://www.ncei.noaa.gov/pub/data/cirs/climdiv/', 
                 headers={'User-Agent': 'CasCrop/1.0'}, timeout=30)
import re
dates = re.findall(r'climdiv-tmaxcy-v[\d.]+-([\d]+)', r.text)
date_stamp = max(dates) if dates else '20260305'
print(f"Using date stamp: {date_stamp}")

climate_files = {
    'climdiv_tmax_county.txt': f'climdiv-tmaxcy-v1.0.0-{date_stamp}',
    'climdiv_tmin_county.txt': f'climdiv-tmincy-v1.0.0-{date_stamp}',
    'climdiv_tavg_county.txt': f'climdiv-tmpccy-v1.0.0-{date_stamp}',
    'climdiv_precip_county.txt': f'climdiv-pcpncy-v1.0.0-{date_stamp}',
    'climdiv_pdsi_county.txt': f'climdiv-pdsicy-v1.0.0-{date_stamp}',
    'climdiv_cdd_county.txt': f'climdiv-cddccy-v1.0.0-{date_stamp}',
    'climdiv_hdd_county.txt': f'climdiv-hddccy-v1.0.0-{date_stamp}',
}
base = 'https://www.ncei.noaa.gov/pub/data/cirs/climdiv/'
for local_name, remote_name in climate_files.items():
    download(f'{base}{remote_name}', f'data/raw/weather/{local_name}', local_name)

In [ ]:
%%time
# 1e. NASS bulk crop data
print("=== USDA NASS Crop Production ===")
nass_url = 'https://www.nass.usda.gov/datasets/qs.crops.txt.gz'
nass_gz = Path('data/raw/nass/qs.crops.txt.gz')
if not nass_gz.exists():
    download(nass_url, nass_gz, 'NASS bulk crops (1 GB - this takes a few minutes)')
else:
    print(f"  Already exists: {nass_gz.name} ({nass_gz.stat().st_size/1e9:.1f} GB)")

# Extract corn/soy/wheat county data
import pandas as pd
filtered_path = Path('data/raw/nass/crops_county_filtered.tsv')
if not filtered_path.exists():
    print("  Extracting corn/soy/wheat county records...")
    target_commodities = {'CORN', 'SOYBEANS', 'WHEAT'}
    target_stats = {'YIELD', 'PRODUCTION', 'AREA PLANTED', 'AREA HARVESTED'}
    count, kept = 0, 0
    with gzip.open(nass_gz, 'rt', encoding='latin-1') as fin, \
         open(filtered_path, 'w') as fout:
        header = fin.readline().strip()
        fout.write(header + '\n')
        cols = header.split('\t')
        ci = cols.index('COMMODITY_DESC')
        si = cols.index('STATISTICCAT_DESC')
        ai = cols.index('AGG_LEVEL_DESC')
        for line in fin:
            count += 1
            parts = line.strip().split('\t')
            if len(parts) <= max(ci, si, ai): continue
            if parts[ci].strip() in target_commodities and parts[si].strip() in target_stats and parts[ai].strip() == 'COUNTY':
                fout.write(line)
                kept += 1
            if count % 5_000_000 == 0:
                print(f"    Processed {count:,} / kept {kept:,}")
    print(f"  Extracted {kept:,} county crop records from {count:,} total")
else:
    print(f"  Already exists: {filtered_path.name}")

In [ ]:
# 1f. Process NASS into clean CSV with waste proxy
nass_clean = Path('data/raw/nass/all_crops_county_annual.csv')
if not nass_clean.exists():
    print("Processing NASS data into clean CSV...")
    df = pd.read_csv(filtered_path, sep='\t', low_memory=False,
                     dtype={'STATE_ANSI': str, 'COUNTY_ANSI': str})
    df['FIPS'] = df['STATE_ANSI'].str.zfill(2) + df['COUNTY_ANSI'].str.zfill(3)
    df = df[df['YEAR'] >= 2008].copy()
    df['VALUE_CLEAN'] = pd.to_numeric(
        df['VALUE'].astype(str).str.replace(',', '').str.strip(),
        errors='coerce'
    )
    
    # Pivot to wide format per county-crop-year
    pivot = df.pivot_table(
        index=['FIPS', 'STATE_ANSI', 'STATE_ALPHA', 'STATE_NAME', 
               'COUNTY_ANSI', 'COUNTY_NAME', 'YEAR', 'COMMODITY_DESC'],
        columns='STATISTICCAT_DESC',
        values='VALUE_CLEAN',
        aggfunc='first'
    ).reset_index()
    pivot.columns.name = None
    
    rename = {
        'COMMODITY_DESC': 'crop',
        'AREA HARVESTED': 'area_harvested_acres',
        'AREA PLANTED': 'area_planted_acres',
        'PRODUCTION': 'production_bu',
        'YIELD': 'yield_bu_per_acre',
    }
    pivot.rename(columns=rename, inplace=True)
    
    # Compute waste proxy
    pivot['waste_proxy'] = (
        (pivot['area_planted_acres'] - pivot['area_harvested_acres']) / 
        pivot['area_planted_acres']
    ).clip(lower=0)
    
    pivot.to_csv(nass_clean, index=False)
    print(f"  Saved: {len(pivot):,} records")
else:
    print(f"Already exists: {nass_clean.name}")

In [ ]:
# Data inventory
print("=== DATA INVENTORY ===")
for source in ['rma', 'nass', 'weather', 'prices', 'geographic']:
    d = Path(f'data/raw/{source}')
    if d.exists():
        files = list(d.rglob('*'))
        files = [f for f in files if f.is_file()]
        size = sum(f.stat().st_size for f in files)
        print(f"  {source:12s}: {len(files):3d} files, {size/1e6:8.1f} MB")

## 2. Process Data + Build Graphs (~5 min)

In [ ]:
%%time
# Run the processing pipeline
!python scripts/02_process_data.py --threshold 100000

In [ ]:
%%time
# Build county graphs
!python scripts/03_build_graphs.py --top-k 20

In [ ]:
# Verify processed data
import pandas as pd, json
features = pd.read_parquet('data/processed/features.parquet')
labels = pd.read_parquet('data/processed/labels.parquet')
with open('data/processed/splits.json') as f: splits = json.load(f)
with open('data/processed/feature_groups.json') as f: groups = json.load(f)

print(f"Samples:    {len(features):,}")
print(f"Counties:   {features['fips'].nunique():,}")
print(f"Waste rate: {labels['waste'].mean():.1%}")
print(f"Features:   bio={len(groups['biophysical'])}, econ={len(groups['economic'])}, hist={len(groups['historical'])}")
print(f"Train/Val/Test: {len(splits['train']):,}/{len(splits['val']):,}/{len(splits['test']):,}")

## 3. Main Ablation Experiment (5 models × 5 seeds × 200 epochs)

This is the core experiment. ~4-6 hours on T4 GPU.

| Row | Model | Tests |
|-----|-------|-------|
| 1 | Local Only (Bio MLP) | Baseline — current SOTA approach |
| 2 | Local + Economic | Adding price features helps? |
| 3 | Geographic GAT | Adding graph structure helps? |
| 4 | Symmetric ECMP | Shock conditioning helps? |
| 5 | **CasCrop (Asymmetric ECMP)** | **Asymmetric shock matters?** |

In [ ]:
%%time
# Run full ablation — this is the big one
!python scripts/04_train_all.py \
    --epochs 200 \
    --patience 20 \
    --batch-size 512 \
    --lr 0.001 \
    --seeds 42 123 456 789 1024 \
    --gpu 0

In [ ]:
# Quick check of results
import json
with open('results/training_results.json') as f:
    results = json.load(f)
print(f"Total runs: {len(results)}")

import pandas as pd
df = pd.DataFrame(results)
for model in ['local_only', 'local_econ', 'geo_gat', 'symmetric_ecmp', 'cascrop']:
    mdf = df[df['model'] == model]
    print(f"{model:20s} AUC={mdf['test_auc_roc'].mean():.3f}±{mdf['test_auc_roc'].std():.3f}  "
          f"F1={mdf['test_f1'].mean():.3f}±{mdf['test_f1'].std():.3f}")

## 4. Extra Experiments

### 4a. Graph Perturbation Test
Shuffle economic edges to prove the graph structure itself matters (not just extra parameters).

In [ ]:
%%time
import numpy as np
import torch
import json
from pathlib import Path

# Load the graph and shuffle edges (keep degree distribution, break structure)
graph_data = np.load('data/graphs/combined_graph.npz')
edge_index = graph_data['edge_index'].copy()
edge_weight = graph_data['edge_weight'].copy()

# Shuffle: randomly permute the target nodes
np.random.seed(42)
shuffled_targets = np.random.permutation(edge_index[1])
edge_index_shuffled = np.array([edge_index[0], shuffled_targets])

# Save shuffled graph
np.savez('data/graphs/combined_graph_shuffled.npz',
         edge_index=edge_index_shuffled, edge_weight=edge_weight)

# Temporarily swap graph, train CasCrop with shuffled edges
import shutil
shutil.copy('data/graphs/combined_graph.npz', 'data/graphs/combined_graph_backup.npz')
shutil.copy('data/graphs/combined_graph_shuffled.npz', 'data/graphs/combined_graph.npz')

!python scripts/04_train_all.py --models cascrop --seeds 42 123 456 --epochs 200 --patience 20 --gpu 0

# Read shuffled results
with open('results/training_results.json') as f:
    shuffled_results = json.load(f)

# Restore original graph
shutil.copy('data/graphs/combined_graph_backup.npz', 'data/graphs/combined_graph.npz')

# Save shuffled results separately
with open('results/graph_perturbation_results.json', 'w') as f:
    json.dump(shuffled_results, f, indent=2)

sdf = pd.DataFrame(shuffled_results)
sdf = sdf[sdf['model'] == 'cascrop']
print(f"\nGraph Perturbation Results:")
print(f"  Shuffled graph AUC: {sdf['test_auc_roc'].mean():.3f} ± {sdf['test_auc_roc'].std():.3f}")
print(f"  Original graph AUC: see main ablation results above")
print(f"  If shuffled << original → graph structure matters (not just parameters)")

### 4b. Edge Type Ablation
Train CasCrop with only geographic edges, only commodity edges, and both.

In [ ]:
%%time
from scipy import sparse

# Load individual adjacency matrices
geo = sparse.load_npz('data/graphs/adjacency_geo.npz')

edge_ablation_results = {}

def sparse_to_topk(matrix, k=20):
    """Convert sparse matrix to top-K edge_index + edge_weight."""
    dense = matrix.toarray()
    n = dense.shape[0]
    rows, cols, vals = [], [], []
    for i in range(n):
        neighbors = dense[i]
        if neighbors.sum() == 0: continue
        top_k_idx = np.argsort(neighbors)[-k:]
        for j in top_k_idx:
            if neighbors[j] > 0:
                rows.append(i); cols.append(j); vals.append(neighbors[j])
    return np.array([rows, cols]), np.array(vals)

# Normalize
def norm_sparse(m):
    mx = m.max()
    return m / mx if mx > 0 else m

geo_norm = norm_sparse(geo)

configs = {
    'geo_only': geo_norm,
    'commodity_only': norm_sparse(
        (sparse.load_npz('data/graphs/adjacency_commodity_corn.npz') +
         sparse.load_npz('data/graphs/adjacency_commodity_soybeans.npz') +
         sparse.load_npz('data/graphs/adjacency_commodity_wheat.npz')) / 3
    ),
}

for config_name, matrix in configs.items():
    print(f"\n--- Edge ablation: {config_name} ---")
    ei, ew = sparse_to_topk(matrix, k=20)
    np.savez('data/graphs/combined_graph.npz', edge_index=ei, edge_weight=ew)
    
    !python scripts/04_train_all.py --models cascrop --seeds 42 123 456 --epochs 200 --patience 20 --gpu 0
    
    with open('results/training_results.json') as f:
        r = json.load(f)
    rdf = pd.DataFrame(r)
    rdf = rdf[rdf['model'] == 'cascrop']
    edge_ablation_results[config_name] = {
        'auc_mean': rdf['test_auc_roc'].mean(),
        'auc_std': rdf['test_auc_roc'].std(),
    }
    print(f"  AUC: {rdf['test_auc_roc'].mean():.3f} ± {rdf['test_auc_roc'].std():.3f}")

# Restore original combined graph
shutil.copy('data/graphs/combined_graph_backup.npz', 'data/graphs/combined_graph.npz')

# Save edge ablation results
with open('results/edge_ablation_results.json', 'w') as f:
    json.dump(edge_ablation_results, f, indent=2)

print("\n=== Edge Type Ablation Summary ===")
for name, r in edge_ablation_results.items():
    print(f"  {name:20s}: AUC = {r['auc_mean']:.3f} ± {r['auc_std']:.3f}")

### 4c. Disentanglement Verification (Linear Probe)

In [ ]:
%%time
import importlib
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Load best CasCrop model
sys.path.insert(0, 'src')
from models.cascrop import CasCrop

# Load processed data
features = pd.read_parquet('data/processed/features.parquet')
labels = pd.read_parquet('data/processed/labels.parquet')
with open('data/processed/splits.json') as f: splits = json.load(f)
with open('data/processed/stats.json') as f: stats = json.load(f)
with open('data/processed/feature_groups.json') as f: groups = json.load(f)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Build test tensors
test_idx = splits['test']
test_feat = features.iloc[test_idx]

def normalize_cols(df, cols, stats):
    X = df[cols].values.astype(np.float32)
    for i, col in enumerate(cols):
        if col in stats:
            X[:, i] = (X[:, i] - stats[col]['mean']) / stats[col]['std']
    return torch.from_numpy(np.nan_to_num(X, 0.0))

x_bio = normalize_cols(test_feat, groups['biophysical'], stats)
x_econ = normalize_cols(test_feat, groups['economic'], stats)
x_hist = normalize_cols(test_feat, groups['historical'], stats)

graph_data = np.load('data/graphs/combined_graph.npz')
edge_index = torch.from_numpy(graph_data['edge_index']).long()

# Load model
ckpt_path = 'checkpoints/cascrop_seed42.pt'
if Path(ckpt_path).exists():
    model = CasCrop(
        bio_input_dim=len(groups['biophysical']),
        econ_input_dim=len(groups['economic']),
        hist_dim=len(groups['historical']),
        latent_dim=64, num_heads=4, dropout=0.0,
    ).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    # Extract z_bio and z_econ
    with torch.no_grad():
        batch = {
            'x_bio': x_bio.to(device),
            'x_econ': x_econ.to(device),
            'x_hist': x_hist.to(device),
            'edge_index': torch.stack([torch.arange(len(x_bio)), torch.arange(len(x_bio))]).to(device),
            'edge_attr': None,
            'price_shocks': torch.zeros(len(x_bio), 1).to(device),
        }
        outputs = model(batch)
        z_bio = outputs['z_bio'].cpu().numpy()
        z_econ = outputs['z_econ'].cpu().numpy()

    # Linear probe: can we predict z_econ clusters from z_bio?
    kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
    econ_labels = kmeans.fit_predict(z_econ)
    
    scaler = StandardScaler()
    z_bio_scaled = scaler.fit_transform(z_bio)
    
    half = len(z_bio) // 2
    probe = LogisticRegression(max_iter=1000, random_state=42)
    probe.fit(z_bio_scaled[:half], econ_labels[:half])
    probe_acc = probe.score(z_bio_scaled[half:], econ_labels[half:])
    
    print(f"=== Disentanglement Verification ===")
    print(f"Linear probe accuracy: {probe_acc:.3f}")
    print(f"Random baseline (5 clusters): 0.200")
    print(f"Target (well disentangled): < 0.550")
    print(f"Result: {'PASS — well disentangled' if probe_acc < 0.55 else 'MARGINAL'}")
    
    # Save
    np.save('results/z_bio_test.npy', z_bio)
    np.save('results/z_econ_test.npy', z_econ)
    with open('results/disentanglement_results.json', 'w') as f:
        json.dump({'linear_probe_accuracy': probe_acc, 'target': 0.55, 'random_baseline': 0.2}, f, indent=2)
else:
    print(f"Checkpoint not found at {ckpt_path}. Run main ablation first.")

## 5. Evaluation + Statistical Tests

In [ ]:
# Restore main ablation results (may have been overwritten by edge ablation)
# Re-run main ablation evaluation
!python scripts/04_train_all.py --epochs 200 --patience 20 --seeds 42 123 456 789 1024 --gpu 0

In [ ]:
# Generate evaluation outputs: tables, figures, statistical tests
!python scripts/05_evaluate_and_publish.py

In [ ]:
# Per-crop subgroup analysis
with open('results/training_results.json') as f:
    all_results = json.load(f)

# Reload main training results and compute per-crop metrics
# (The test predictions are stored in checkpoints)
print("=== Per-Crop Waste Rates ===")
for crop in ['CORN', 'SOYBEANS', 'WHEAT']:
    crop_labels = labels[labels['commodity'] == crop]
    print(f"  {crop}: {crop_labels['waste'].mean():.1%} waste rate ({len(crop_labels):,} samples)")

print("\n=== Per-Cause Distribution ===")
print(labels['cause_category'].value_counts().to_string())

### 5a. Full Statistical Significance Tests

In [ ]:
from evaluation.statistical_tests import paired_ttest_across_seeds, wilcoxon_test_across_seeds

df = pd.DataFrame(all_results)
cascrop_aucs = df[df['model'] == 'cascrop']['test_auc_roc'].tolist()

print("=== Statistical Significance Tests (CasCrop vs each baseline) ===")
print(f"{'Comparison':<35} {'ΔAUC':>8} {'t-stat':>8} {'p-value':>10} {'Sig':>6}")
print("-" * 75)

comparisons = {}
for model in ['local_only', 'local_econ', 'geo_gat', 'symmetric_ecmp']:
    model_aucs = df[df['model'] == model]['test_auc_roc'].tolist()
    if len(model_aucs) != len(cascrop_aucs):
        continue
    
    t = paired_ttest_across_seeds(cascrop_aucs, model_aucs)
    w = wilcoxon_test_across_seeds(cascrop_aucs, model_aucs)
    
    sig = '***' if t['p_value'] < 0.001 else '**' if t['p_value'] < 0.01 else '*' if t['p_value'] < 0.05 else 'n.s.'
    
    print(f"CasCrop vs {model:<23} {t['mean_diff']:>+.4f} {t['t_statistic']:>8.3f} {t['p_value']:>10.6f} {sig:>6}")
    comparisons[f'cascrop_vs_{model}'] = {
        'paired_ttest': t,
        'wilcoxon': w,
    }

with open('results/statistical_tests.json', 'w') as f:
    json.dump(comparisons, f, indent=2, default=str)

## 6. Publication Figures

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 9, 'font.family': 'sans-serif', 'figure.dpi': 300})

# Figure 3: Main Ablation Bar Chart
model_order = ['local_only', 'local_econ', 'geo_gat', 'symmetric_ecmp', 'cascrop']
display = ['Row 1:\nLocal Only', 'Row 2:\nLocal+Econ', 'Row 3:\nGeo GAT', 'Row 4:\nSymmetric', 'Row 5:\nCasCrop']
colors = ['#7f8c8d', '#3498db', '#e67e22', '#9b59b6', '#e74c3c']

means, stds = [], []
for m in model_order:
    mdf = df[df['model'] == m]
    means.append(mdf['test_auc_roc'].mean())
    stds.append(mdf['test_auc_roc'].std())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Panel A: AUC-ROC
x = np.arange(len(model_order))
bars = ax1.bar(x, means, 0.6, yerr=stds, capsize=4, color=colors,
               edgecolor='black', linewidth=0.5)
for bar, val, std in zip(bars, means, stds):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.005,
             f'{val:.3f}', ha='center', va='bottom', fontsize=7)
ax1.set_ylabel('AUC-ROC')
ax1.set_xticks(x)
ax1.set_xticklabels(display, fontsize=7)
ax1.set_ylim(0.7, 1.0)
ax1.grid(axis='y', alpha=0.3)
ax1.set_title('(a) Test Set AUC-ROC', fontweight='bold')

# Panel B: AUC-PR
means_pr, stds_pr = [], []
for m in model_order:
    mdf = df[df['model'] == m]
    means_pr.append(mdf['test_auc_pr'].mean())
    stds_pr.append(mdf['test_auc_pr'].std())

bars2 = ax2.bar(x, means_pr, 0.6, yerr=stds_pr, capsize=4, color=colors,
                edgecolor='black', linewidth=0.5)
for bar, val, std in zip(bars2, means_pr, stds_pr):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.005,
             f'{val:.3f}', ha='center', va='bottom', fontsize=7)
ax2.set_ylabel('AUC-PR')
ax2.set_xticks(x)
ax2.set_xticklabels(display, fontsize=7)
ax2.set_ylim(0.5, 1.0)
ax2.grid(axis='y', alpha=0.3)
ax2.set_title('(b) Test Set AUC-PR', fontweight='bold')

plt.tight_layout()
fig.savefig('paper/figures/fig3_ablation.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: paper/figures/fig3_ablation.pdf')

In [ ]:
# Figure 6: Disentanglement t-SNE
from sklearn.manifold import TSNE

z_bio_path = Path('results/z_bio_test.npy')
z_econ_path = Path('results/z_econ_test.npy')

if z_bio_path.exists() and z_econ_path.exists():
    z_bio = np.load(z_bio_path)
    z_econ = np.load(z_econ_path)
    
    n = min(3000, len(z_bio))
    idx = np.random.choice(len(z_bio), n, replace=False)
    
    test_features = features.iloc[splits['test']]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    
    tsne1 = TSNE(n_components=2, random_state=42, perplexity=30)
    z1_2d = tsne1.fit_transform(z_bio[idx])
    
    # Color by temperature (biophysical)
    if 'tavg' in test_features.columns:
        c1 = test_features['tavg'].iloc[idx].fillna(0).values
    else:
        c1 = np.random.randn(n)
    sc1 = ax1.scatter(z1_2d[:, 0], z1_2d[:, 1], c=c1, cmap='coolwarm', s=1, alpha=0.5, rasterized=True)
    ax1.set_title('(a) z_bio colored by temperature', fontweight='bold', fontsize=9)
    plt.colorbar(sc1, ax=ax1, label='Avg Temperature')
    
    tsne2 = TSNE(n_components=2, random_state=42, perplexity=30)
    z2_2d = tsne2.fit_transform(z_econ[idx])
    
    if 'price_mean' in test_features.columns:
        c2 = test_features['price_mean'].iloc[idx].fillna(0).values
    else:
        c2 = np.random.randn(n)
    sc2 = ax2.scatter(z2_2d[:, 0], z2_2d[:, 1], c=c2, cmap='RdYlGn_r', s=1, alpha=0.5, rasterized=True)
    ax2.set_title('(b) z_econ colored by price level', fontweight='bold', fontsize=9)
    plt.colorbar(sc2, ax=ax2, label='Commodity Price')
    
    plt.tight_layout()
    fig.savefig('paper/figures/fig6_disentanglement.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved: paper/figures/fig6_disentanglement.pdf')
else:
    print('Run disentanglement verification first (Section 4c)')

## 7. Generate LaTeX Tables

In [ ]:
# Table 2: Main ablation table (the central table of the paper)
display_names = {
    'local_only': 'Row 1: Local Only (Bio MLP)',
    'local_econ': 'Row 2: Local + Economic',
    'geo_gat': 'Row 3: Geographic GAT',
    'symmetric_ecmp': 'Row 4: Symmetric ECMP',
    'cascrop': 'Row 5: Full CasCrop$^\\dagger$',
}

metrics = ['test_auc_roc', 'test_f1', 'test_auc_pr']
metric_labels = ['AUC-ROC', 'F1', 'AUC-PR']

# Find best for bolding
best_vals = {m: df.groupby('model')[m].mean().max() for m in metrics}

latex_rows = []
for model in model_order:
    mdf = df[df['model'] == model]
    name = display_names[model]
    cells = [name]
    for metric in metrics:
        mean = mdf[metric].mean()
        std = mdf[metric].std()
        cell = f'{mean:.3f} $\\pm$ {std:.3f}'
        if mean == best_vals[metric]:
            cell = f'\\textbf{{{cell}}}'
        cells.append(cell)
    cells.append(f"{mdf['n_params'].iloc[0]:,}")
    latex_rows.append(' & '.join(cells) + ' \\\\')

latex = """\\begin{table*}[t]
\\centering
\\caption{Main ablation results on test set (2022--2024). Values are mean $\\pm$ std across 5 seeds. 
\\textbf{Bold}: best. $^\\dagger$: asymmetric ECMP with disentanglement.}
\\label{tab:ablation}
\\begin{tabular}{lcccc}
\\toprule
Model & AUC-ROC & F1 & AUC-PR & Params \\\\
\\midrule
""" + '\n'.join(latex_rows) + """
\\bottomrule
\\end{tabular}
\\end{table*}"""

with open('paper/tables/table2_ablation.tex', 'w') as f:
    f.write(latex)
print(latex)
print('\nSaved: paper/tables/table2_ablation.tex')

## 8. Hypothesis Verification Summary

In [ ]:
cascrop_auc = df[df['model'] == 'cascrop']['test_auc_roc'].mean()
local_auc = df[df['model'] == 'local_only']['test_auc_roc'].mean()
econ_auc = df[df['model'] == 'local_econ']['test_auc_roc'].mean()
geo_auc = df[df['model'] == 'geo_gat']['test_auc_roc'].mean()
sym_auc = df[df['model'] == 'symmetric_ecmp']['test_auc_roc'].mean()

print("=" * 60)
print("HYPOTHESIS VERIFICATION")
print("=" * 60)
print(f"")
print(f"H1: Graph models > Independent models")
print(f"    CasCrop ({cascrop_auc:.3f}) vs Local Only ({local_auc:.3f}): Δ = +{cascrop_auc-local_auc:.3f}")
print(f"    {'✓ CONFIRMED' if cascrop_auc > local_auc else '✗ FAILED'}")
print(f"")
print(f"H2: Economic edges > Geographic-only edges")
print(f"    CasCrop ({cascrop_auc:.3f}) vs Geo GAT ({geo_auc:.3f}): Δ = +{cascrop_auc-geo_auc:.3f}")
print(f"    {'✓ CONFIRMED' if cascrop_auc > geo_auc else '✗ FAILED'}")
print(f"")
print(f"H3: Asymmetric > Symmetric shock conditioning")
print(f"    CasCrop ({cascrop_auc:.3f}) vs Symmetric ({sym_auc:.3f}): Δ = +{cascrop_auc-sym_auc:.3f}")
print(f"    {'✓ CONFIRMED' if cascrop_auc > sym_auc else '✗ FAILED'}")
print(f"")
print(f"Economic features alone (no graph):")
print(f"    Local+Econ ({econ_auc:.3f}) vs Local Only ({local_auc:.3f}): Δ = +{econ_auc-local_auc:.3f}")

# Save summary
summary = {
    'H1_graph_vs_independent': {'delta': cascrop_auc - local_auc, 'confirmed': cascrop_auc > local_auc},
    'H2_econ_vs_geo': {'delta': cascrop_auc - geo_auc, 'confirmed': cascrop_auc > geo_auc},
    'H3_asymmetric_vs_symmetric': {'delta': cascrop_auc - sym_auc, 'confirmed': cascrop_auc > sym_auc},
    'model_aucs': {m: df[df['model']==m]['test_auc_roc'].mean() for m in model_order},
}
with open('results/hypothesis_verification.json', 'w') as f:
    json.dump(summary, f, indent=2)

## 9. Download Results

Download all outputs for your paper.

In [ ]:
# Package all results for download
import shutil

!tar czf cascrop_results.tar.gz results/ paper/figures/ paper/tables/ checkpoints/
print("Packaged: cascrop_results.tar.gz")
print("Download this file — it contains all results, figures, tables, and model checkpoints.")

# In Colab, use:
try:
    from google.colab import files
    files.download('cascrop_results.tar.gz')
except ImportError:
    print("Not in Colab — download manually from the file browser.")

In [ ]:
# Final summary
print("=" * 60)
print("ALL EXPERIMENTS COMPLETE")
print("=" * 60)
print(f"")
print(f"Generated outputs:")
print(f"  results/training_results.json          — All ablation metrics")
print(f"  results/statistical_tests.json          — DeLong, t-test, Wilcoxon p-values")
print(f"  results/hypothesis_verification.json    — H1/H2/H3 confirmed/failed")
print(f"  results/disentanglement_results.json    — Linear probe accuracy")
print(f"  results/graph_perturbation_results.json — Shuffled graph performance")
print(f"  results/edge_ablation_results.json      — Geo-only vs commodity-only")
print(f"  paper/figures/fig3_ablation.pdf          — Main ablation bar chart")
print(f"  paper/figures/fig6_disentanglement.pdf   — t-SNE disentanglement")
print(f"  paper/tables/table2_ablation.tex         — LaTeX ablation table")
print(f"  checkpoints/*.pt                         — Trained model weights")
print(f"")
print(f"Next: Drop these into your paper/main.tex and submit.")